In [1]:
import pandas as pd
import nltk.data
import json
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from multiprocessing import Pool

#nltk.download('punkt')
sent_detector = nltk.data.load('tokenizers/punkt/english.pickle')

In [2]:
df_filings = pd.read_csv("../data/10K_filings.csv")
df_filings.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription,ticker,cik,text
0,0000320193-22-000108,2022-10-28,2022-09-24,2022-10-27T18:01:14.000Z,34.0,10-K,001-36743,221338448,NaN,10332356,1,1,aapl-20220924.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20220924.htm\n 10-K\n \n \n \n...
1,0000320193-21-000105,2021-10-29,2021-09-25,2021-10-28T18:04:28.000Z,34.0,10-K,001-36743,211359752,NaN,10502096,1,1,aapl-20210925.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20210925.htm\n 10-K\n \n \n \n...
2,0000320193-20-000096,2020-10-30,2020-09-26,2020-10-29T18:06:25.000Z,34.0,10-K,001-36743,201273977,NaN,12502600,1,1,aapl-20200926.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20200926.htm\n 10-K\n \n \n \n...
3,0000320193-19-000119,2019-10-31,2019-09-28,2019-10-30T18:12:36.000Z,34.0,10-K,001-36743,191181423,NaN,12861616,1,1,a10-k20199282019.htm,10-K,AAPL,320193,10-K\n 1\n a10-k20199282019.htm\n 10-K\n \n \n...
4,0000320193-18-000145,2018-11-05,2018-09-29,2018-11-05T08:01:40.000Z,34.0,10-K,001-36743,181158788,NaN,12275572,1,0,a10-k20189292018.htm,10-K,AAPL,320193,10-K\n 1\n a10-k20189292018.htm\n 10-K\n \n \n...


In [3]:
sentences = sent_detector.tokenize(df_filings.text[0].strip())
len(sentences)

970

In [4]:
import pandas as pd
import nltk.data
sent_detector = nltk.data.load('tokenizers/punkt/english.pickle')
import json
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# FinBERT model
finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-esg',num_labels=4)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-esg')
nlp = pipeline("text-classification", model=finbert, tokenizer=tokenizer)

In [5]:

# download necessary filings data
df_filings = pd.read_csv("../data/10K_filings.csv")
df_filings.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription,ticker,cik,text
0,0000320193-22-000108,2022-10-28,2022-09-24,2022-10-27T18:01:14.000Z,34.0,10-K,001-36743,221338448,NaN,10332356,1,1,aapl-20220924.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20220924.htm\n 10-K\n \n \n \n...
1,0000320193-21-000105,2021-10-29,2021-09-25,2021-10-28T18:04:28.000Z,34.0,10-K,001-36743,211359752,NaN,10502096,1,1,aapl-20210925.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20210925.htm\n 10-K\n \n \n \n...
2,0000320193-20-000096,2020-10-30,2020-09-26,2020-10-29T18:06:25.000Z,34.0,10-K,001-36743,201273977,NaN,12502600,1,1,aapl-20200926.htm,10-K,AAPL,320193,10-K\n 1\n aapl-20200926.htm\n 10-K\n \n \n \n...
3,0000320193-19-000119,2019-10-31,2019-09-28,2019-10-30T18:12:36.000Z,34.0,10-K,001-36743,191181423,NaN,12861616,1,1,a10-k20199282019.htm,10-K,AAPL,320193,10-K\n 1\n a10-k20199282019.htm\n 10-K\n \n \n...
4,0000320193-18-000145,2018-11-05,2018-09-29,2018-11-05T08:01:40.000Z,34.0,10-K,001-36743,181158788,NaN,12275572,1,0,a10-k20189292018.htm,10-K,AAPL,320193,10-K\n 1\n a10-k20189292018.htm\n 10-K\n \n \n...


In [6]:
import torch

device = torch.device("mps")
# FinBERT model
finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-esg',num_labels=4)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-esg')
#nlp = pipeline("text-classification", model=finbert, tokenizer=tokenizer)

inputs    = tokenizer(sentences[:32], return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)
finbert   = finbert.to(device)
outputs   = finbert(**inputs)

In [11]:
device = torch.device("mps")

In [5]:
report = df_filings.text[0]
filings_sentences = sent_detector.tokenize(report.strip())
report_sentences = pd.DataFrame(filings_sentences, columns = ["sentence"])
#report_sentences.loc[:, "id"] = list(range(report_sentences.shape[0]))
report_sentences

,sentence
0,10-K\n 1\n aapl-20220924.htm\n 10-K\n \n \n \n...
1,Commission File Number: 001-36743 Apple Inc. ...
2,Employer Identification No.)
3,"One Apple Park Way Cupertino , California 950..."
4,Yes ☒ No ☐ Indicate by check mark if...
...,...
965,Apple Inc. | 2022 Form 10-K | 57 SIGNATURES Pu...
966,"Date: October 27, 2022 Apple Inc. By: /s/ Luca..."
967,Pursuant to the requirements of the Securities...
968,"Bell Director October 27, 2022 JAMES A."


In [11]:
def esg_sentence_disclosure(sentence, idx):    
    try:
        result = nlp(sentence)
        if not(result[0]["label"] == "None"):
            esg_disclosure[result[0]["label"]]['sentence_id'].append(idx)
            sentences_tmp = report_sentences.loc[max(idx-1, 0): min(report_sentences.shape[0], idx + 1)].sentence.values.tolist()
            esg_disclosure[result[0]["label"]]['content'].append(" ".join(sentences_tmp))
    except:
        #nbr_of_sentences_with_more_than_max_tokens += 1
        pass

report = df_filings.text[0]
filings_sentences = sent_detector.tokenize(report.strip())
report_sentences = pd.DataFrame(filings_sentences, columns = ["sentence"])
report_sentences.loc[:, "id"] = list(range(report_sentences.shape[0]))

esg_disclosure = {}
esg_disclosure['Environmental'] = {}
esg_disclosure['Environmental']["sentence_id"] = []
esg_disclosure['Environmental']["content"] = []
esg_disclosure['Social'] = {}
esg_disclosure['Social']["sentence_id"] = []
esg_disclosure['Social']["content"] = []
esg_disclosure['Governance'] = {}
esg_disclosure['Governance']["sentence_id"] = []
esg_disclosure['Governance']["content"] = []

nbr_of_sentences_with_more_than_max_tokens = 0

for idx in range(report_sentences.shape[0]):
    esg_sentence_disclosure(report_sentences.loc[idx, "sentence"], idx)


In [12]:
esg_disclosure

{'Environmental': {'sentence_id': [122, 168, 370, 371, 372, 373, 374],
  'content': ['The Company periodically provides certain information for investors on its corporate website, www.apple.com, and its investor relations website, investor.apple.com. This includes press releases and other information about financial performance, information on environmental, social and governance matters, and details related to the Company’s annual meeting of shareholders. The information contained on the websites referenced in this Form 10-K is not incorporated by reference into this filing.',
   'In addition, such operations and facilities are subject to the risk of interruption by fire, power shortages, nuclear power plant accidents and other industrial accidents, terrorist attacks and other hostile acts, ransomware and other cybersecurity attacks, labor disputes, public health issues, including pandemics such as the COVID-19 pandemic, and other events beyond the Company’s control. Global climate ch

In [14]:
len(df_filings.text[0].strip().split(" "))

34650

In [10]:
from multiprocessing import Pool

def esg_sentence_disclosure(sentence, idx):    
    try:
        result = nlp(sentence)
        if not(result[0]["label"] == "None"):
            esg_disclosure[result[0]["label"]]['sentence_id'].append(idx)
            sentences_tmp = report_sentences.loc[max(idx-1, 0): min(report_sentences.shape[0], idx + 1)].sentence.values.tolist()
            esg_disclosure[result[0]["label"]]['content'].append(" ".join(sentences_tmp))
    except:
        #nbr_of_sentences_with_more_than_max_tokens += 1
        pass


report = df_filings.text[0]
filings_sentences = sent_detector.tokenize(report.strip())
report_sentences = pd.DataFrame(filings_sentences, columns = ["sentence"])
report_sentences.loc[:, "id"] = list(range(report_sentences.shape[0]))

esg_disclosure = {}
esg_disclosure['Environmental'] = {}
esg_disclosure['Environmental']["sentence_id"] = []
esg_disclosure['Environmental']["content"] = []
esg_disclosure['Social'] = {}
esg_disclosure['Social']["sentence_id"] = []
esg_disclosure['Social']["content"] = []
esg_disclosure['Governance'] = {}
esg_disclosure['Governance']["sentence_id"] = []
esg_disclosure['Governance']["content"] = []

with Pool(4) as pool:
    pool.starmap(esg_sentence_disclosure, zip(report_sentences.sentence.tolist(), report_sentences.index.tolist()))



Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'esg_sentence_disclosure' on <module '__main__' (built-in)>
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootst

KeyboardInterrupt: 

In [14]:



def examine_esg_disclosure(report, filename):
    filings_sentences = sent_detector.tokenize(report.strip())
    report_sentences = pd.DataFrame(filings_sentences, columns = ["sentence"])
    report_sentences.loc[:, "id"] = list(range(report_sentences.shape[0]))

    esg_disclosure = {}
    esg_disclosure['Environmental'] = {}
    esg_disclosure['Environmental']["sentence_id"] = []
    esg_disclosure['Environmental']["content"] = []
    esg_disclosure['Social'] = {}
    esg_disclosure['Social']["sentence_id"] = []
    esg_disclosure['Social']["content"] = []
    esg_disclosure['Governance'] = {}
    esg_disclosure['Governance']["sentence_id"] = []
    esg_disclosure['Governance']["content"] = []

    nbr_of_sentences_with_more_than_max_tokens = 0

    for idx, row in report_sentences.iterrows():
        try:
            result = nlp(row["sentence"])
            if not(result[0]["label"] == "None"):
                esg_disclosure[result[0]["label"]]['sentence_id'].append(idx)
                sentences_tmp = report_sentences.loc[max(idx-1, 0): min(report_sentences.shape[0], idx + 1)].sentence.values.tolist()
                esg_disclosure[result[0]["label"]]['content'].append(" ".join(sentences_tmp))
        except:
            #nbr_of_sentences_with_more_than_max_tokens += 1
            pass

    nbr_of_environmental_sentences = len(esg_disclosure["Environmental"]["sentence_id"])
    nbr_of_social_sentences = len(esg_disclosure["Social"]["sentence_id"])
    nbr_of_governance_sentences = len(esg_disclosure["Governance"]["sentence_id"])

    nbr_of_labeled_sentences = report_sentences.shape[0] - nbr_of_sentences_with_more_than_max_tokens


    esg_disclosure["Summary"] = {}
    esg_disclosure["Summary"]["nbr_of_labeled_sentences"] = nbr_of_labeled_sentences
    esg_disclosure["Summary"]["nbr_of_environmental_sentences"] = nbr_of_environmental_sentences
    esg_disclosure["Summary"]["nbr_of_social_sentences"] = nbr_of_social_sentences
    esg_disclosure["Summary"]["nbr_of_governance_sentences"] = nbr_of_governance_sentences


    with open(f"{filename}.json", "w") as fp:
        json.dump(esg_disclosure, fp)

,sentence,id
0,10-K\n 1\n aapl-20220924.htm\n 10-K\n \n \n \n...,0
1,Commission File Number: 001-36743 Apple Inc. ...,1
2,Employer Identification No.),2
3,"One Apple Park Way Cupertino , California 950...",3
4,Yes ☒ No ☐ Indicate by check mark if...,4
...,...,...
965,Apple Inc. | 2022 Form 10-K | 57 SIGNATURES Pu...,965
966,"Date: October 27, 2022 Apple Inc. By: /s/ Luca...",966
967,Pursuant to the requirements of the Securities...,967
968,"Bell Director October 27, 2022 JAMES A.",968


In [4]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-esg',num_labels=4)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-esg')
nlp = pipeline("text-classification", model=finbert, tokenizer=tokenizer)

In [32]:
from tqdm.notebook import tqdm


esg_disclosure = {}
esg_disclosure['Environmental'] = {}
esg_disclosure['Environmental']["sentence_id"] = []
esg_disclosure['Environmental']["content"] = []
esg_disclosure['Social'] = {}
esg_disclosure['Social']["sentence_id"] = []
esg_disclosure['Social']["content"] = []
esg_disclosure['Governance'] = {}
esg_disclosure['Governance']["sentence_id"] = []
esg_disclosure['Governance']["content"] = []

nbr_of_sentences_with_more_than_max_tokens = 0

for idx, row in tqdm(report_sentences.iterrows()):
    #if row["is_in_token_range"]:
    try:
        result = nlp(row["sentence"])
        if not(result[0]["label"] == "None"):
            esg_disclosure[result[0]["label"]]['sentence_id'].append(idx)
            sentences_tmp = report_sentences.loc[max(idx-1, 0): min(report_sentences.shape[0], idx + 1)].sentence.values.tolist()
            esg_disclosure[result[0]["label"]]['content'].append(" ".join(sentences_tmp))
    except:
        nbr_of_sentences_with_more_than_max_tokens += 1

nbr_of_environmental_sentences = len(esg_disclosure["Environmental"]["sentence_id"])
nbr_of_social_sentences = len(esg_disclosure["Social"]["sentence_id"])
nbr_of_governance_sentences = len(esg_disclosure["Governance"]["sentence_id"])

nbr_of_labeled_sentences = report_sentences.shape[0] - nbr_of_sentences_with_more_than_max_tokens


esg_disclosure["Summary"] = {}
esg_disclosure["Summary"]["nbr_of_labeled_sentences"] = nbr_of_labeled_sentences
esg_disclosure["Summary"]["nbr_of_environmental_sentences"] = nbr_of_environmental_sentences
esg_disclosure["Summary"]["nbr_of_social_sentences"] = nbr_of_social_sentences
esg_disclosure["Summary"]["nbr_of_governance_sentences"] = nbr_of_governance_sentences

0it [00:00, ?it/s]

In [36]:
import json

with open('data.json', 'w') as fp:
    json.dump(esg_disclosure, fp)

In [40]:
dict_json["Environmental"]

{'sentence_id': [122, 168, 370, 371, 372, 373, 374],
 'content': ['The Company periodically provides certain information for investors on its corporate website, www.apple.com, and its investor relations website, investor.apple.com. This includes press releases and other information about financial performance, information on environmental, social and governance matters, and details related to the Company’s annual meeting of shareholders. The information contained on the websites referenced in this Form 10-K is not incorporated by reference into this filing.',
  'In addition, such operations and facilities are subject to the risk of interruption by fire, power shortages, nuclear power plant accidents and other industrial accidents, terrorist attacks and other hostile acts, ransomware and other cybersecurity attacks, labor disputes, public health issues, including pandemics such as the COVID-19 pandemic, and other events beyond the Company’s control. Global climate change is resulting in

In [39]:
import json

with open("data.json", "r") as fp:
    dict_json = json.load(fp)
dict_json

{'Environmental': {'sentence_id': [122, 168, 370, 371, 372, 373, 374],
  'content': ['The Company periodically provides certain information for investors on its corporate website, www.apple.com, and its investor relations website, investor.apple.com. This includes press releases and other information about financial performance, information on environmental, social and governance matters, and details related to the Company’s annual meeting of shareholders. The information contained on the websites referenced in this Form 10-K is not incorporated by reference into this filing.',
   'In addition, such operations and facilities are subject to the risk of interruption by fire, power shortages, nuclear power plant accidents and other industrial accidents, terrorist attacks and other hostile acts, ransomware and other cybersecurity attacks, labor disputes, public health issues, including pandemics such as the COVID-19 pandemic, and other events beyond the Company’s control. Global climate ch

In [26]:
for sentenes in esg_disclosure["Environmental"]["content"]:
    print("-")
    print(sentenes)
    print("-")

-
The Company periodically provides certain information for investors on its corporate website, www.apple.com, and its investor relations website, investor.apple.com. This includes press releases and other information about financial performance, information on environmental, social and governance matters, and details related to the Company’s annual meeting of shareholders. The information contained on the websites referenced in this Form 10-K is not incorporated by reference into this filing.
-
-
In addition, such operations and facilities are subject to the risk of interruption by fire, power shortages, nuclear power plant accidents and other industrial accidents, terrorist attacks and other hostile acts, ransomware and other cybersecurity attacks, labor disputes, public health issues, including pandemics such as the COVID-19 pandemic, and other events beyond the Company’s control. Global climate change is resulting in certain types of natural disasters occurring more frequently or w

In [48]:
esg_disclosure[result[0]["label"]]

KeyError: 'None'